In [45]:
import os

In [46]:
%pwd

'c:\\Users\\godje\\OneDrive\\Escritorio\\Data-Science-Project-Stroke-Prediction-'

In [4]:
os.chdir('../')

In [47]:
%pwd

'c:\\Users\\godje\\OneDrive\\Escritorio\\Data-Science-Project-Stroke-Prediction-'

In [30]:
import pandas as pd
data = pd.read_csv('artifacts/data_ingestion/healthcare-dataset-stroke-data.csv')
data.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [31]:
data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [32]:
data.isnull().sum()

id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

In [33]:
df.shape

(5110, 12)

In [82]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataValidationConfig:
    root_dir: Path
    unzip_data_dir: Path
    STATUS_FILE: str
    all_schema: dict


In [83]:
from src.stroke_prediction.constants import *
from src.stroke_prediction.utils.common import read_yaml, create_directories

In [84]:
class ConfiguartionManager:
    def __init__(self,
                 config_file_path = CONFIG_FILE_PATH,
                 params_file_path = PARAMS_FILE_PATH,
                 schema_file_path = SCHEMA_FILE_PATH): 
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root_dir])


    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir = config.root_dir,
            unzip_data_dir = config.unzip_data_dir,
            STATUS_FILE = config.STATUS_FILE,
            all_schema = schema
        )

        return data_validation_config

In [85]:
import os
from src.stroke_prediction import logger

ModuleNotFoundError: No module named 'src'

In [86]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self)-> bool:
        try:
            validated_status = True
            
            data = pd.read_csv(self.config.unzip_data_dir)
            all_cols = list(data.columns)
            all_schema = self.config.all_schema.keys()

            for col in all_cols:
                if col not in all_schema:
                    validated_status = False
                    logger.info(f"Data Validation Failed. Column: {col} is not in the schema.")
                    break

            with open(self.config.STATUS_FILE, 'w') as f:
                f.write(f"Validation Status: {validated_status}")

            return validated_status

        except Exception as e:
            raise e


    def validate_data_types(self) -> bool:
        try:
            validated_status = True

            data = pd.read_csv(self.config.unzip_data_dir)
            all_schema = self.config.all_schema

            for col in data.columns:
                if col in all_schema:
                    expected_dtype = all_schema[col]
                    actual_dtype = str(data[col].dtype)

                    if expected_dtype != actual_dtype:
                        validated_status = False
                        logger.info(f"Data Validation Failed. Column: {col} has dtype {actual_dtype} but expected {expected_dtype}.")
                        break
            with open(self.config.STATUS_FILE, 'w') as f:
                f.write(f"Validation Status: {validated_status}")
            return validated_status

        except Exception as e:
            raise e

In [87]:
try:
    config = ConfiguartionManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_columns()
    data_validation.validate_data_types()
    logger.info("Data Validation Completed.")
except Exception as e:
    logger.exception(e)

2026-05-08 01:17:49,309 - INFO - Data Validation Completed.
